# Распарсим ссылки на другие документы внутри 845-П

In [7]:
import csv
import xml.etree.ElementTree as ET
import zipfile
from urllib.parse import urlparse

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Пространства имён ODF и XLink
NS = {
    'text':  'urn:oasis:names:tc:opendocument:xmlns:text:1.0',
    'xlink': 'http://www.w3.org/1999/xlink',
}

def extract_links(odt_path, external_only=True):
    """
    Извлекает все гиперссылки из ODT-файла.

    :param odt_path: путь к .odt
    :param external_only: если True — только внешние ссылки (http/https),
                          иначе включая внутренние (#anchor) и mailto.
    :return: список словарей {'url': ..., 'text': ...}
    """
    links = []
    with zipfile.ZipFile(odt_path, 'r') as z, z.open('content.xml') as f:
        tree = ET.parse(f)
        root = tree.getroot()

        # .//text:a — все элементы <text:a> на любой глубине
        for a in root.findall('.//text:a', NS):
            href = a.get('{http://www.w3.org/1999/xlink}href')
            if not href:
                continue

            # Собираем ВЕСЬ текст внутри <text:a>, включая вложенные span'ы
            text = ''.join(a.itertext()).strip()

            if external_only:
                scheme = urlparse(href).scheme
                if scheme not in ('http', 'https'):
                    continue

            links.append({'url': href, 'text': text})
    return links


def dedupe(links):
    """Убирает дубликаты по URL, сохраняя первое вхождение."""
    seen, result = set(), []
    for ln in links:
        if ln['url'] not in seen:
            seen.add(ln['url'])
            result.append(ln)
    return result

In [30]:
path = "../data/raw/845_p.odt"
links = extract_links(path)

for i, ln in enumerate(links, 1):
    print(f"{i}. {ln['url']}\n   текст: {ln['text']}\n")
    # Убираем указание параграфа, чтобы сохранять только адрес документа
    # ln['url'] = ln['url'][:ln['url'].rfind("/")] 
    # Только айдишник документа в Гарант
    ln['url'] = ln['url'].split("/")[-2]

# Сохранение в CSV (удобно открыть в Excel)
with open('../data/raw/links.csv', 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['url', 'text'], delimiter=';')
    w.writeheader()
    w.writerows(links)
print(f"Всего уникальных ссылок: {len(links)} → сохранено в links.csv")

1. http://ivo.garant.ru/document/redirect/411309161/0
   текст: Положение Банка России от 2 ноября 2024 г. N 845-П
"О порядке расчета величины кредитного риска банками с применением банковских методик управления кредитным риском и моделей количественной оценки кредитного риска"

2. http://ivo.garant.ru/document/redirect/12127405/7204
   текст: части четвертой статьи 72

3. http://ivo.garant.ru/document/redirect/12127405/721
   текст: частей первой

4. http://ivo.garant.ru/document/redirect/12127405/7214
   текст: четвертой

5. http://ivo.garant.ru/document/redirect/12127405/7216
   текст: шестой

6. http://ivo.garant.ru/document/redirect/12127405/72111
   текст: одиннадцатой статьи 72 1

7. http://ivo.garant.ru/document/redirect/12127405/7212
   текст: частью второй

8. http://ivo.garant.ru/document/redirect/12127405/72111
   текст: частью одиннадцатой

9. http://ivo.garant.ru/document/redirect/12127405/721
   текст: статьи 72 1

10. http://ivo.garant.ru/document/redirect/12127405/623


# Прочитаем айдишники документов и выгрузим документы из Гарант

In [8]:
links = pd.read_csv("../data/raw/links.csv", sep = ";")

In [9]:
document_ids = list(links["url"].unique())

In [49]:
def download_odt(topic_id, token):
    # Строго фиксированный URL по спецификации Гарант API 2.3.0
    url = f"https://api.garant.ru/v2/topic/{topic_id}/download-odt" # html
    
    headers = {
        "Authorization": f"Bearer {token}"
    }
    
    print(f"Отправка запроса на URL: {url}") # Для самопроверки в консоли
    
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status()
        
        filename = f"../data/raw/document_{topic_id}.odt"
        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
        print(f"Успешно скачано! Файл: {filename}")
        
    except requests.exceptions.HTTPError as e:
        print(f"Ошибка сервера Гарант: {e} (Код: {response.status_code})")
        if response.status_code == 401:
            print("Проверьте ваш токен авторизации.")
        elif response.status_code == 404:
            print(f"Документ с ID {topic_id} не найден в системе.")
    except Exception as e:
        print(f"Сетевая или системная ошибка: {e}")

# --- ЗАПУСК ---
# Замените только значения внутри кавычек
MY_TOKEN = "4fba8a34ce1b11f1b77a0050568d72f0" # Перенесите сюда ваш токен

In [50]:
for id in tqdm(document_ids):
    id = str(id)
    print(id)
    download_odt(topic_id=id, token=MY_TOKEN)
    break

  0%|          | 0/92 [00:00<?, ?it/s]

411309161
Отправка запроса на URL: https://api.garant.ru/v2/topic/411309161/download-odt
Ошибка сервера Гарант: 423 Client Error:  for url: https://api.garant.ru/v2/topic/411309161/download-odt (Код: 423)


In [16]:
from pathlib import Path

# Все объекты в папке (файлы и папки)
p = Path('../data/raw')
files = [str(item) for item in p.iterdir() if item.is_file()]

In [17]:
files

['../data/raw/12112327.odt',
 '../data/raw/72042602.odt',
 '../data/raw/document_107917.odt',
 '../data/raw/75093468.odt',
 '../data/raw/400794152.odt',
 '../data/raw/document_72051916.odt',
 '../data/raw/74279372.odt',
 '../data/raw/document_71057396.odt',
 '../data/raw/10105800.odt',
 '../data/raw/12127405.odt',
 '../data/raw/document_406750235.odt',
 '../data/raw/72158318.odt',
 '../data/raw/406741957.odt',
 '../data/raw/411309161.odt',
 '../data/raw/406821239.odt',
 '../data/raw/71323300.odt',
 '../data/raw/document_12154854.odt',
 '../data/raw/185181.odt',
 '../data/raw/400697241.odt',
 '../data/raw/document_75093468.odt',
 '../data/raw/document_400794152.odt',
 '../data/raw/12166036.odt',
 '../data/raw/71127972.odt',
 '../data/raw/406693531.odt',
 '../data/raw/406750235.odt',
 '../data/raw/411309162.odt',
 '../data/raw/405233105.odt',
 '../data/raw/404461100.odt',
 '../data/raw/845_p.odt',
 '../data/raw/document_71323300.odt',
 '../data/raw/72051916.odt',
 '../data/raw/73081370.o

In [15]:
downloaded_ids = [s[s.find("_") + 1:s.rfind(".")] for s in files]

In [22]:
s = '../data/raw/document_107917.odt'

In [23]:
s[s.find("_") + 1:s.rfind(".")]

'107917'

In [12]:
downloaded_ids


['../data/raw/12112327',
 '../data/raw/72042602',
 '107917',
 '../data/raw/75093468',
 '../data/raw/400794152',
 '72051916',
 '../data/raw/74279372',
 '71057396',
 '../data/raw/10105800',
 '../data/raw/12127405',
 '406750235',
 '../data/raw/72158318',
 '../data/raw/406741957',
 '../data/raw/411309161',
 '../data/raw/406821239',
 '../data/raw/71323300',
 '12154854',
 '../data/raw/185181',
 '../data/raw/400697241',
 '75093468',
 '400794152',
 '../data/raw/12166036',
 '../data/raw/71127972',
 '../data/raw/406693531',
 '../data/raw/406750235',
 '../data/raw/411309162',
 '../data/raw/405233105',
 '../data/raw/404461100',
 'p',
 '71323300',
 '../data/raw/72051916',
 '../data/raw/73081370',
 '584330',
 '../data/raw/409466821',
 '12127405',
 '../data/raw/72141792',
 '../data/raw/403304818',
 '../data/raw/404461098',
 '10105800',
 '../data/raw/73853102',
 '../data/raw/406061235',
 '../data/raw/71457074',
 '400697241',
 '../data/raw/400697303',
 '12148517',
 '../data/raw/407438021',
 '../data/ra

In [13]:
document_ids

[np.int64(411309161),
 np.int64(12127405),
 np.int64(72051916),
 np.int64(71283076),
 np.int64(73363119),
 np.int64(406061235),
 np.int64(72077620),
 np.int64(408664973),
 np.int64(71127772),
 np.int64(107917),
 np.int64(12112327),
 np.int64(12138267),
 np.int64(71323300),
 np.int64(71926818),
 np.int64(400697241),
 np.int64(584330),
 np.int64(400794152),
 np.int64(406741957),
 np.int64(406750235),
 np.int64(75093468),
 np.int64(12148517),
 np.int64(12154854),
 np.int64(71721612),
 np.int64(71901656),
 np.int64(71057396),
 np.int64(413275699),
 np.int64(10105800),
 np.int64(10164324),
 np.int64(185181),
 np.int64(411309162),
 np.int64(71203444),
 np.int64(71292204),
 np.int64(72286130),
 np.int64(73853102),
 np.int64(73972093),
 np.int64(400697303),
 np.int64(400924741),
 np.int64(402617260),
 np.int64(407030464),
 np.int64(74279372),
 np.int64(71127972),
 np.int64(72141792),
 np.int64(72803576),
 np.int64(74479859),
 np.int64(407438021),
 np.int64(72193696),
 np.int64(404461100),
 np.

In [9]:
to_download = list(set(document_ids) - set(downloaded_ids))

In [15]:
dosc = [str(x) for x in to_download]

In [17]:
for d in dosc:
    print(f"https://ivo.garant.ru/#/document/{d}")

https://ivo.garant.ru/#/document/74650624
https://ivo.garant.ru/#/document/408455187
https://ivo.garant.ru/#/document/73081370
https://ivo.garant.ru/#/document/406750235
https://ivo.garant.ru/#/document/407812637
https://ivo.garant.ru/#/document/71926818
https://ivo.garant.ru/#/document/400794152
https://ivo.garant.ru/#/document/404461098
https://ivo.garant.ru/#/document/404461100
https://ivo.garant.ru/#/document/413275699
https://ivo.garant.ru/#/document/400924741
https://ivo.garant.ru/#/document/408029781
https://ivo.garant.ru/#/document/73853020
https://ivo.garant.ru/#/document/10164324
https://ivo.garant.ru/#/document/411309161
https://ivo.garant.ru/#/document/411309162
https://ivo.garant.ru/#/document/72042602
https://ivo.garant.ru/#/document/72158318
https://ivo.garant.ru/#/document/408453743
https://ivo.garant.ru/#/document/71203444
https://ivo.garant.ru/#/document/72071298
https://ivo.garant.ru/#/document/584330
https://ivo.garant.ru/#/document/71721612
https://ivo.garant.ru/#/

In [22]:
from selenium import webdriver

# Инициализируем драйвер браузера (например, Chrome)
driver = webdriver.Chrome()


for d in dosc:
    print(f"https://ivo.garant.ru/#/document/{d}")
    # Открываем нужную ссылку
    driver.get("https://ivo.garant.ru/#/document/{d}")

https://ivo.garant.ru/#/document/74650624
https://ivo.garant.ru/#/document/408455187
https://ivo.garant.ru/#/document/73081370
https://ivo.garant.ru/#/document/406750235
https://ivo.garant.ru/#/document/407812637
https://ivo.garant.ru/#/document/71926818
https://ivo.garant.ru/#/document/400794152
https://ivo.garant.ru/#/document/404461098
https://ivo.garant.ru/#/document/404461100
https://ivo.garant.ru/#/document/413275699
https://ivo.garant.ru/#/document/400924741
https://ivo.garant.ru/#/document/408029781
https://ivo.garant.ru/#/document/73853020
https://ivo.garant.ru/#/document/10164324
https://ivo.garant.ru/#/document/411309161
https://ivo.garant.ru/#/document/411309162
https://ivo.garant.ru/#/document/72042602
https://ivo.garant.ru/#/document/72158318
https://ivo.garant.ru/#/document/408453743
https://ivo.garant.ru/#/document/71203444
https://ivo.garant.ru/#/document/72071298
https://ivo.garant.ru/#/document/584330
https://ivo.garant.ru/#/document/71721612
https://ivo.garant.ru/#/

In [20]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys

# Инициализируем драйвер
driver = webdriver.Chrome()
driver.get("https://ivo.garant.ru/#/document/71057396")
time.sleep(2)  # Даем странице полностью загрузиться

# 1. Фокусируемся на странице (кликаем в пустое место)
driver.find_element(By.TAG_NAME, "body").click()

# 2. Зажимаем Ctrl, жмем S, отпускаем Ctrl, ждем долю секунды и жмем Enter
actions = ActionChains(driver)
(
    actions.key_down(Keys.CONTROL)
    .send_keys('s')
    .key_up(Keys.CONTROL)
    .pause(0.5)        # Небольшая пауза, чтобы сайт успел среагировать на Ctrl+S
    .send_keys(Keys.ENTER)
    .perform()
)
